[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JMasr/ai-tutorials/blob/main/es/terminal-10-90-lab.ipynb)

Author: José Manuel Ramírez Sánchez · https://jmramirez.engineer

# El 10 % de la terminal: laboratorio con cuatro flags

Acompaña al post [El 10 % de la terminal que te resuelve el 90 %](https://jmramirez.engineer/es/writing/terminal-10-90/), que explica cada habilidad con animaciones. Aquí las practicas en una máquina de verdad: cada sección esconde una flag, y `check()` al final te dice cuántas tienes.

La historia es la del post: son las 2:47, el entrenamiento nocturno de Whisper en el servidor `gpu` ha muerto con `CUDA out of memory`, y te toca resucitarlo. Ten a mano [la chuleta](https://jmramirez.engineer/cheatsheets/terminal-10-90-es.pdf).

### Dónde se ejecuta

Este notebook está hecho para **Google Colab** (gratis; basta con CPU). Instala paquetes, crea un usuario llamado `username` y arranca un servidor SSH que solo escucha dentro de la máquina virtual de Colab, en `127.0.0.1`. Esa máquina es tuya, temporal, y se borra al desconectar.

No lo ejecutes en tu ordenador ni en un servidor compartido: crea un usuario con una contraseña conocida. Si aun así quieres hacerlo, usa una máquina virtual desechable, bajo tu responsabilidad. El notebook se ofrece tal cual, sin garantías (licencia MIT).

### El escenario

| En el post | En este laboratorio |
|---|---|
| tu portátil | la terminal de esta máquina de Colab (eres `root`, tu carpeta es `~`) |
| el servidor `gpu`, `10.0.3.7` | el usuario `username` en `127.0.0.1`, puerto `2222`, en la misma máquina |
| el kernel de Jupyter olvidado | un proceso que se come un núcleo de CPU (Colab gratis no tiene GPU) |

Como el «servidor» vive en la misma máquina, su dirección es `127.0.0.1` y necesita `-p 2222` o `Port 2222`. Todo lo demás es exactamente igual que en un servidor de verdad.

## 0. Prepara el laboratorio

Ejecuta esta celda (unos 30 s). Apunta la **huella** que imprime: la necesitarás en la sección 2.

In [ ]:
# Lab setup. Run it once; running it again starts the lab from scratch with new flags.
# Only in Google Colab: it installs packages, creates the user `username` and starts an SSH server
# that listens on 127.0.0.1 only, inside this disposable virtual machine.
import base64, hashlib, json, os, secrets, shutil, socket, subprocess, sys, textwrap, time
from pathlib import Path

M = {'proj': 'proyectos', 'dl': 'Descargas', 'notes': 'notas.txt', 'note_text': 'whisper: entrenamiento nocturno en gpu, revisar por la mañana\n', 'colab_only': "Este laboratorio solo se ejecuta en Google Colab: instala paquetes, crea un usuario y arranca un servidor SSH. Ábrelo con el botón «Open in Colab». (En una máquina virtual desechable, bajo tu responsabilidad: os.environ['LAB_OUTSIDE_COLAB'] = '1'.)", 'installing': 'Instalando openssh-server, tmux y nano (unos 30 s)…', 'ready': 'Laboratorio listo.\n  portátil: esta máquina de Colab, tu carpeta {laptop}\n  servidor "gpu": usuario {user} en 127.0.0.1, puerto {port} (solo accesible desde dentro de esta máquina)\n  contraseña de {user}, solo para ssh-copy-id: {password}\n  huella de gpu: {fp}\n\nSiguiente: abre la terminal con la celda de abajo.', 'dry_oom': 'batch_size: {} -> CUDA out of memory (GPU de 24 GB). Bájalo a 16 en configs/train.yaml.', 'dry_ok': 'config OK (batch_size: {}) · {}', 'oom': 'torch.OutOfMemoryError: CUDA out of memory (batch_size: {}). ¿Está el config corregido en esta máquina?', 'no_tmux': 'Corriendo fuera de una sesión de tmux llamada "train": si se corta la conexión, muere contigo.\n(La flag solo aparece dentro de `tmux new -s train`.)', 'done': 'entrenamiento terminado mientras no estabas · {}', 'noflag': 'entrenamiento terminado. Sin flag: lánzalo dentro de la sesión "train" de tmux y sal con Ctrl-b d mientras corre.', 'ghost': 'Algo se está comiendo un núcleo de CPU en gpu. Se para solo en {} min.', 'setup_first': 'Ejecuta primero la celda de preparación.', 'hints': ['Moverte: corrige el config con nano y ejecuta train.py --dry-run en el portátil.', 'SSH: tráete logs/train.log de gpu al portátil con scp.', 'tmux: entrena dentro de la sesión "train" y sal con Ctrl-b d mientras corre.', 'Monitorización: encuentra el proceso que se come la CPU en gpu y termínalo.'], 'ok': 'hecho', 'all': 'Cuatro de cuatro. Son las 3:20: a dormir.', 'nolog': 'La flag es correcta, pero train.log no está en el portátil: tráetelo con scp.', 'alive': 'El nombre es correcto, pero el proceso sigue vivo: termínalo con kill.'}
USER, PASSWORD, PORT = 'username', 'gpu-lab', 2222
LAPTOP = Path.home()                    # the "laptop": your folder in this Colab machine
PROJ = LAPTOP / M['proj'] / 'whisper'
SERVER = Path('/home') / USER           # the "server" gpu: another user, reached through SSH
STATE = LAPTOP / '.lab-10-90'           # flag hashes (reading them gives nothing away)
TRAIN_PY = '#!/usr/bin/env python3\n"""Whisper training (a simulation for the lab). Reads configs/train.yaml."""\nimport base64, os, subprocess, sys, time\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nunhide = lambda s: base64.b64decode(s[::-1]).decode()\ncfg = (HERE / \'configs\' / \'train.yaml\').read_text()\nbatch = int(next(l.split(\':\')[1] for l in cfg.splitlines() if l.strip().startswith(\'batch_size\')))\n\nif \'--dry-run\' in sys.argv:\n    if batch > 16:\n        sys.exit(@DRY_OOM@.format(batch))\n    print(@DRY_OK@.format(batch, unhide(@F1@)))\n    sys.exit()\n\nif batch > 16:\n    print(\'epoch 1/12 · loss 2.31\', flush=True)\n    time.sleep(2)\n    sys.exit(@OOM@.format(batch))\n\ndef session():\n    if not os.environ.get(\'TMUX\'):\n        return None\n    return subprocess.run([\'tmux\', \'display-message\', \'-p\', \'#S\'], capture_output=True, text=True).stdout.strip()\n\ndef attached():\n    out = subprocess.run([\'tmux\', \'list-clients\', \'-t\', \'train\'], capture_output=True, text=True).stdout\n    return bool(out.strip())\n\nname = session()\nif name != \'train\':\n    print(@NO_TMUX@, flush=True)\ndetached = False\nfor epoch in range(1, 13):\n    for _ in range(10):\n        time.sleep(1)\n        if name == \'train\' and not attached():\n            detached = True\n    print(f\'epoch {epoch}/12 · loss {2.31 * 0.82 ** epoch:.2f}\', flush=True)\nmsg = @DONE@.format(unhide(@F3@)) if name == \'train\' and detached else @NOFLAG@\nprint(msg)\nwith open(HERE / \'logs\' / \'last-run.txt\', \'a\') as f:   # every run adds its line\n    f.write(msg + \'\\n\')\n'
TRAIN_YAML = 'model: openai/whisper-small\nlanguage: es\nbatch_size: 32\nlearning_rate: 1.0e-5\n'
OLD_LOG = """[02:41:07] run {run} · model openai/whisper-small · batch_size 32
[02:41:09] epoch 1/12 · loss 2.31
[02:46:52] epoch 2/12 · loss 1.87
[02:47:03] torch.OutOfMemoryError: CUDA out of memory. Tried to allocate 1.20 GiB
[02:47:03] GPU 0 has a total capacity of 23.99 GiB of which 412.00 MiB is free.
[02:47:03] Process 2290 has 9.30 GiB memory in use.
[02:47:03] run {run} · status FAILED · keep this file: {flag}
"""

if 'google.colab' not in sys.modules and os.environ.get('LAB_OUTSIDE_COLAB') != '1':
    raise RuntimeError(M['colab_only'])


def run(cmd, check=True):
    return subprocess.run(cmd, shell=True, check=check, text=True, capture_output=True)


def write(path, text, mode=0o644):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
    path.chmod(mode)


sha = lambda s: hashlib.sha256(str(s).strip().encode()).hexdigest()
hide = lambda s: base64.b64encode(s.encode()).decode()[::-1]   # only so that grep does not find the flags


def setup():
    print(M['installing'])
    run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openssh-server tmux nano procps > /dev/null')
    run('service ssh stop', check=False)                     # the default server on port 22 is not needed
    run('pip install -q colab-xterm', check=False)
    run(f'id -u {USER} || useradd -m -s /bin/bash {USER}')
    run(f'echo "{USER}:{PASSWORD}" | chpasswd')
    run(f'pkill -u {USER}', check=False)                     # leftovers of a previous run
    for p in (PROJ.parent, SERVER / 'whisper', SERVER / '.ssh', SERVER / '.local', STATE):
        shutil.rmtree(p, ignore_errors=True)

    flags = {f'FLAG{i}': f'FLAG{{{secrets.token_hex(4)}}}' for i in (1, 2, 3)}
    flags['FLAG4'] = f'ghost-{secrets.token_hex(3)}'
    code = TRAIN_PY.replace('@F1@', repr(hide(flags['FLAG1']))).replace('@F3@', repr(hide(flags['FLAG3'])))
    for k in ('dry_oom', 'dry_ok', 'oom', 'no_tmux', 'done', 'noflag'):
        code = code.replace(f'@{k.upper()}@', repr(M[k]))

    # The laptop: a project with the config that broke the night's training.
    for root in (PROJ, SERVER / 'whisper'):
        write(root / 'train.py', code, 0o755)
        write(root / 'configs' / 'train.yaml', TRAIN_YAML)
        (root / 'data').mkdir(exist_ok=True)
    (LAPTOP / M['dl']).mkdir(exist_ok=True)
    write(LAPTOP / M['notes'], M['note_text'])
    # The server: the log of the failed run, and a prompt that says where you are.
    write(SERVER / 'whisper' / 'logs' / 'train.log', OLD_LOG.format(run=secrets.token_hex(3), flag=flags['FLAG2']))
    for home, host in ((SERVER, 'gpu'), (LAPTOP, 'laptop')):
        rc = home / '.bashrc'
        text = rc.read_text() if rc.exists() else ''
        if '# lab-10-90' not in text:
            rc.write_text(text + f"\n# lab-10-90\nPS1='\\u@{host}:\\w\\$ '\n")
    (SERVER / '.ssh').mkdir(mode=0o700)
    run(f'chown -R {USER}: {SERVER}')
    STATE.mkdir(mode=0o700)
    (STATE / 'flags.json').write_text(json.dumps({k: sha(v) for k, v in flags.items()}))
    (STATE / 'ghost').write_text(flags['FLAG4'])

    # The SSH server of gpu: its own configuration, on 127.0.0.1:2222 only.
    run('mkdir -p /run/sshd && ssh-keygen -A')
    write('/etc/ssh/sshd_lab_config', textwrap.dedent(f"""\
        Port {PORT}
        ListenAddress 127.0.0.1
        HostKey /etc/ssh/ssh_host_ed25519_key
        PidFile /run/sshd-lab.pid
        AllowUsers {USER}
        PermitRootLogin no
        PubkeyAuthentication yes
        PasswordAuthentication yes
        KbdInteractiveAuthentication no
        UsePAM no
        AllowTcpForwarding no
        X11Forwarding no
        Subsystem sftp /usr/lib/openssh/sftp-server
        """))
    run('kill "$(cat /run/sshd-lab.pid)"', check=False)
    time.sleep(0.5)
    run('/usr/sbin/sshd -f /etc/ssh/sshd_lab_config')
    for _ in range(20):
        with socket.socket() as s:
            if s.connect_ex(('127.0.0.1', PORT)) == 0:
                break
        time.sleep(0.25)
    fp = run('ssh-keygen -lf /etc/ssh/ssh_host_ed25519_key.pub').stdout.split()[1]
    print(M['ready'].format(laptop=LAPTOP, user=USER, port=PORT, password=PASSWORD, fp=fp))


def ghost(minutes=10):
    """Start the process that eats the CPU on gpu: one core, lowest priority, gone after `minutes`."""
    if not (STATE / 'ghost').exists():
        raise RuntimeError(M['setup_first'])
    name = (STATE / 'ghost').read_text().strip()
    binary = SERVER / '.local' / 'bin' / name
    binary.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(shutil.which('yes'), binary)
    run(f'chown -R {USER}: {SERVER / ".local"}')
    subprocess.Popen(['runuser', '-u', USER, '--', 'sh', '-c',
                      f'nice -n 19 "{binary}" > /dev/null & g=$!; sleep {minutes * 60}; kill $g 2>/dev/null'],
                     start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(M['ghost'].format(minutes))


def check(*answers):
    """Check the flags you found, in any order: check('FLAG{...}', 'ghost-...')."""
    if not (STATE / 'flags.json').exists():
        raise RuntimeError(M['setup_first'])
    hashes = json.loads((STATE / 'flags.json').read_text())
    found = {k for a in answers for k, h in hashes.items() if sha(a) == h}
    notes = dict(zip(sorted(hashes), M['hints']))
    if 'FLAG2' in found and not (list(PROJ.rglob('train.log')) or (LAPTOP / 'train.log').exists()):
        found.discard('FLAG2'); notes['FLAG2'] = M['nolog']
    name = (STATE / 'ghost').read_text().strip()
    if 'FLAG4' in found and run(f'pgrep -x {name}', check=False).returncode == 0:
        found.discard('FLAG4'); notes['FLAG4'] = M['alive']
    for i, k in enumerate(sorted(hashes), 1):
        print(f'{"✓" if k in found else "·"} {i}. {M["ok"] if k in found else notes[k]}')
    if len(found) == 4:
        print('\n' + M['all'])


setup()

## Abre la terminal

La celda siguiente abre una terminal debajo, con [colab-xterm](https://github.com/InfuseAI/colab-xterm). A partir de aquí todo se escribe **ahí**, no en celdas. La terminal sigue abierta mientras ejecutas otras celdas.

Si no se abre, Colab Pro tiene su propio botón **Terminal** abajo a la izquierda.

In [ ]:
%load_ext colabxterm
%xterm

## 1. Moverte sin interfaz

Primero, en el portátil: el config del entrenamiento de esta noche tiene `batch_size: 32`, que no cabe en la GPU.

1. Ve a tu carpeta personal, comprueba dónde estás y lista lo que hay.
2. Entra en `proyectos/whisper` y lístalo.
3. Ejecuta `python3 train.py --dry-run`: comprueba el config sin entrenar. Falla.
4. Abre `configs/train.yaml` con `nano`, cambia `32` por `16`, guarda y sal.
5. Repite el dry run. La **flag 1** está en su salida.

<details><summary>Pista</summary>

```bash
cd ~
pwd
ls
cd proyectos/whisper
ls
python3 train.py --dry-run
nano configs/train.yaml      # Ctrl-O y Enter guarda, Ctrl-X sale
python3 train.py --dry-run
```
</details>

## 2. Entrar en el servidor

El config está arreglado en el portátil, pero el entrenamiento vive en `gpu`.

1. Crea tu par de claves con `ssh-keygen -t ed25519`. Pulsa Enter para aceptar la ruta; la passphrase es opcional en este laboratorio.
2. Copia la clave pública al servidor con `ssh-copy-id -p 2222 username@127.0.0.1`. La primera vez enseña la huella del servidor: responde `yes` solo si coincide con la que imprimió la celda de preparación. La contraseña es `gpu-lab`.
3. Dale un nombre corto al servidor en `~/.ssh/config`:

```
Host gpu
    HostName 127.0.0.1
    Port 2222
    User username
```

4. Entra con `ssh gpu`. El prompt cambia a `username@gpu`. Echa un vistazo con `ls` y sal con `exit`.
5. De vuelta en `~/proyectos/whisper` en el portátil, lleva el config corregido al servidor, a `whisper/configs/`.
6. Tráete el log del run que falló, `whisper/logs/train.log`, del servidor al portátil. La **flag 2** está en su última línea. `check()` también quiere el archivo en el portátil.

<details><summary>Pista</summary>

```bash
ssh-keygen -t ed25519
ssh-copy-id -p 2222 username@127.0.0.1
nano ~/.ssh/config
ssh gpu
exit
cd ~/proyectos/whisper
scp configs/train.yaml gpu:whisper/configs/
scp gpu:whisper/logs/train.log .
nano train.log
```
</details>

## 3. Dejar el entrenamiento corriendo: tmux

Ahora relanza el entrenamiento en `gpu`, de forma que sobreviva a que te vayas. Tarda 2 minutos.

1. Entra en `gpu` y en `whisper`.
2. Abre una sesión de tmux llamada `train` y lanza dentro `python3 train.py`. Si falla con `CUDA out of memory`, el config corregido aún no está en el servidor (sección 2).
3. Sal de la sesión con `Ctrl-b d` mientras entrena, y del servidor con `exit`.
4. Espera unos 2 minutos. Luego vuelve a entrar en `gpu` y engánchate a la sesión. La **flag 3** está en la última línea (también queda guardada en `whisper/logs/last-run.txt`).

La flag solo aparece si el entrenamiento corrió dentro de la sesión `train` y saliste de ella mientras corría.

<details><summary>Pista</summary>

```bash
ssh gpu
cd whisper
tmux new -s train
python3 train.py
# Ctrl-b, suelta, d
exit
# ...2 minutos después
ssh gpu
tmux ls
tmux attach -t train
```
</details>

## 4. Ver qué está pasando: top

El entrenamiento corre, pero algo más se está comiendo el servidor. Ejecuta la celda siguiente: arranca en `gpu`, como el usuario `username`, un proceso que mantiene ocupado un núcleo de CPU (se para solo a los 10 minutos).

In [ ]:
ghost()

En la terminal:

1. Entra en `gpu` y abre `top`. Busca el proceso que encabeza `%CPU` y apunta su PID y su nombre (columna `COMMAND`). Sal con `q`.
2. Antes de matarlo, mira de quién es y qué es con `ps -o user=,args= -p PID`.
3. Es tuyo (`username`): termínalo con `kill PID`. Vuelve a abrir `top` para comprobar que ya no está.
4. La **flag 4** es el nombre del proceso, `ghost-` seguido de seis caracteres.

Con un entorno de ejecución con GPU en Colab también puedes ejecutar `nvidia-smi` en la terminal del portátil y ver uno de verdad.

<details><summary>Pista</summary>

```bash
ssh gpu
top                        # q para salir
ps -o user=,args= -p PID   # el PID que viste en top
kill PID
top
```
</details>

## Comprueba tus flags

Pega las cuatro flags en `check()`, en cualquier orden, y ejecuta la celda. Te dice cuáles están bien y qué falta.

In [ ]:
check('FLAG{...}', 'FLAG{...}', 'FLAG{...}', 'ghost-...')

## Y ahora

Has hecho cada habilidad una vez de principio a fin. Repítelo mañana sin las pistas y empezará a salirte de memoria.

- El post con las animaciones: [El 10 % de la terminal que te resuelve el 90 %](https://jmramirez.engineer/es/writing/terminal-10-90/)
- [La chuleta de una página](https://jmramirez.engineer/cheatsheets/terminal-10-90-es.pdf)
- Para empezar de cero, con flags nuevas, vuelve a ejecutar la celda de preparación. Para borrarlo todo, *Entorno de ejecución → Desconectar y eliminar entorno de ejecución*.